In [19]:
import logging
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, days, col, monotonically_increasing_id, sha2, concat_ws
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampNTZType

In [2]:
def create_spark_session(app_name: str) -> SparkSession:
    spark = (
        SparkSession.builder
        .master("spark://spark-master:7077")
        .appName(app_name)
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.lakehouse_prod","org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.lakehouse_prod.type", "hive")
        .config("spark.sql.catalog.lakehouse_prod.uri","thrift://hive-metastore:9083")
        .config("spark.sql.catalog.lakehouse_prod.warehouse","s3a://lakehouse-prod-bucket/warehouse/")
        .config("spark.sql.catalog.lakehouse_prod.io-impl","org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .enableHiveSupport()
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")
    print("NOTE: SparkSession created successfully!")

    return spark

In [3]:
app_name = 'Spark-Iceberg-Test'
spark = create_spark_session(app_name)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 18:43:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


NOTE: SparkSession created successfully!


In [5]:
df = spark.read.parquet("s3a://datalake-landing/currency_profiles/currency_profiles_20260402.parquet", header=True, inferSchema=True)
df.show()

+-------------+------+--------------------+-------------+--------------+--------+----+--------------------+--------------------+
|currency_code|symbol|                name|symbol_native|decimal_digits|rounding|code|         name_plural|       Ingested_Time|
+-------------+------+--------------------+-------------+--------------+--------+----+--------------------+--------------------+
|          USD|     $|           US Dollar|            $|             2|     0.0| USD|          US dollars|2026-04-02 17:05:...|
|          CAD|   CA$|     Canadian Dollar|            $|             2|     0.0| CAD|    Canadian dollars|2026-04-02 17:05:...|
|          EUR|     €|                Euro|            €|             2|     0.0| EUR|               euros|2026-04-02 17:05:...|
|          AED|   AED|United Arab Emira...|        د.إ.‏|             2|     0.0| AED|         UAE dirhams|2026-04-02 17:05:...|
|          AFN|    Af|      Afghan Afghani|            ؋|             0|     0.0| AFN|     Afghan

In [7]:
catalog_name = "lakehouse_prod"
fetched_schema_name = "bronze_db"
fetched_table_name = "bronze_country_iso_profiles"

In [9]:
currency_df = spark.read.format("iceberg").load(f"{catalog_name}.{fetched_schema_name}.{fetched_table_name}")
currency_df.show()

[Stage 5:>                                                          (0 + 1) / 1]

+----+----+-------+-------------------+-------------------+--------------------+
|iso2|iso3|iso_num|            country|     country_common|       Ingested_Time|
+----+----+-------+-------------------+-------------------+--------------------+
|  AF| AFG|      4|        Afghanistan|        Afghanistan|2026-04-01 13:52:...|
|  AX| ALA|    248|      Aland Islands|      Aland Islands|2026-04-01 13:52:...|
|  AL| ALB|      8|            Albania|            Albania|2026-04-01 13:52:...|
|  DZ| DZA|     12|            Algeria|            Algeria|2026-04-01 13:52:...|
|  AS| ASM|     16|     American Samoa|     American Samoa|2026-04-01 13:52:...|
|  AD| AND|     20|            Andorra|            Andorra|2026-04-01 13:52:...|
|  AO| AGO|     24|             Angola|             Angola|2026-04-01 13:52:...|
|  AI| AIA|    660|           Anguilla|           Anguilla|2026-04-01 13:52:...|
|  AQ| ATA|     10|         Antarctica|         Antarctica|2026-04-01 13:52:...|
|  AG| ATG|     28|Antigua a

In [18]:
test_currency_df = currency_df.withColumn(
    "record_hash",
    sha2(concat_ws("||", "iso2", "iso3", "iso_num", "country"), 256)
)

test_currency_df.show()

+----+----+-------+-------------------+-------------------+--------------------+--------------------+
|iso2|iso3|iso_num|            country|     country_common|       Ingested_Time|         record_hash|
+----+----+-------+-------------------+-------------------+--------------------+--------------------+
|  AF| AFG|      4|        Afghanistan|        Afghanistan|2026-04-01 13:52:...|f19c840f19f2a5efc...|
|  AX| ALA|    248|      Aland Islands|      Aland Islands|2026-04-01 13:52:...|a1d40b8cae2bee553...|
|  AL| ALB|      8|            Albania|            Albania|2026-04-01 13:52:...|bcc3e4cdb671aa95e...|
|  DZ| DZA|     12|            Algeria|            Algeria|2026-04-01 13:52:...|18fb4664204274eb8...|
|  AS| ASM|     16|     American Samoa|     American Samoa|2026-04-01 13:52:...|0d018bd7f05526337...|
|  AD| AND|     20|            Andorra|            Andorra|2026-04-01 13:52:...|95a4c01f20874d01f...|
|  AO| AGO|     24|             Angola|             Angola|2026-04-01 13:52:...|e6

In [21]:
test_currency_df = currency_df.withColumn(
    "record_hash",
    monotonically_increasing_id()
)

test_currency_df.show()

+----+----+-------+-------------------+-------------------+--------------------+-----------+
|iso2|iso3|iso_num|            country|     country_common|       Ingested_Time|record_hash|
+----+----+-------+-------------------+-------------------+--------------------+-----------+
|  AF| AFG|      4|        Afghanistan|        Afghanistan|2026-04-02 14:33:...|          0|
|  AX| ALA|    248|      Aland Islands|      Aland Islands|2026-04-02 14:33:...|          1|
|  AL| ALB|      8|            Albania|            Albania|2026-04-02 14:33:...|          2|
|  DZ| DZA|     12|            Algeria|            Algeria|2026-04-02 14:33:...|          3|
|  AS| ASM|     16|     American Samoa|     American Samoa|2026-04-02 14:33:...|          4|
|  AD| AND|     20|            Andorra|            Andorra|2026-04-02 14:33:...|          5|
|  AO| AGO|     24|             Angola|             Angola|2026-04-02 14:33:...|          6|
|  AI| AIA|    660|           Anguilla|           Anguilla|2026-04-02 

**DIM_CURRENCY**
- CURRENCY_KEY
- CURRENCY_NAME
- CURRENCY_ISO_CODE
- CREATED_AT
- UPDATED_AT
- IS_CURRENT_FLAG
- EFFECTIVE_DATE
- EXPIRY_DATE

In [25]:
start_id = 1000
currency_df = currency_df.withColumn("ID", start_id + monotonically_increasing_id())
currency_df.show()

+----+----+-------+-------------------+-------------------+--------------------+----+
|iso2|iso3|iso_num|            country|     country_common|       Ingested_Time|  ID|
+----+----+-------+-------------------+-------------------+--------------------+----+
|  AF| AFG|      4|        Afghanistan|        Afghanistan|2026-04-02 14:33:...|1000|
|  AX| ALA|    248|      Aland Islands|      Aland Islands|2026-04-02 14:33:...|1001|
|  AL| ALB|      8|            Albania|            Albania|2026-04-02 14:33:...|1002|
|  DZ| DZA|     12|            Algeria|            Algeria|2026-04-02 14:33:...|1003|
|  AS| ASM|     16|     American Samoa|     American Samoa|2026-04-02 14:33:...|1004|
|  AD| AND|     20|            Andorra|            Andorra|2026-04-02 14:33:...|1005|
|  AO| AGO|     24|             Angola|             Angola|2026-04-02 14:33:...|1006|
|  AI| AIA|    660|           Anguilla|           Anguilla|2026-04-02 14:33:...|1007|
|  AQ| ATA|     10|         Antarctica|         Antarc

In [31]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lpad, col

window_spec = Window.orderBy("iso2")  # choose a stable column

currency_df = currency_df.withColumn(
    "seq_id",
    lpad(row_number().over(window_spec).cast("string"), 4, "0")
)
currency_df.show()

+----+----+-------+--------------------+--------------------+--------------------+----+------+
|iso2|iso3|iso_num|             country|      country_common|       Ingested_Time|  ID|seq_id|
+----+----+-------+--------------------+--------------------+--------------------+----+------+
|NULL| NAM|    516|             Namibia|             Namibia|2026-04-02 14:33:...|1153|  0001|
|NULL| NAM|    516|             Namibia|             Namibia|2026-04-01 13:52:...|1402|  0002|
|  AD| AND|     20|             Andorra|             Andorra|2026-04-02 14:33:...|1005|  0003|
|  AD| AND|     20|             Andorra|             Andorra|2026-04-01 13:52:...|1254|  0004|
|  AE| ARE|    784|United Arab Emira...|United Arab Emirates|2026-04-02 14:33:...|1233|  0005|
|  AE| ARE|    784|United Arab Emira...|United Arab Emirates|2026-04-01 13:52:...|1482|  0006|
|  AF| AFG|      4|         Afghanistan|         Afghanistan|2026-04-01 13:52:...|1249|  0007|
|  AF| AFG|      4|         Afghanistan|         A

In [34]:
window_spec

In [35]:
currency_df

DataFrame[iso2: string, iso3: string, iso_num: bigint, country: string, country_common: string, Ingested_Time: timestamp, ID: bigint, seq_id: string]

In [ ]:
currency_df.select(
    "",
    
)

In [ ]:
spark.stop()